<a href="https://colab.research.google.com/github/rendyamril/data-science-2026/blob/main/Pertemuan10_MohammadRendyAmril_24040101028.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Nama : Mohammad Rendy Amril
* NIM : 240401010288
* Prodi : S1 PJJ Informatika
* Kelas : IF 403

In [14]:
import pandas as pd

# Mengunduh dataset Telco Customer Churn langsung dari link publik
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Ukuran Data:", df.shape)
print("\nDistribusi Target (Churn):")
print(df["Churn"].value_counts(normalize=True))

Ukuran Data: (7043, 21)

Distribusi Target (Churn):
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


In [15]:
from sklearn.model_selection import train_test_split

# 1. Bersihkan kolom TotalCharges (ubah spasi/string kosong menjadi angka)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 2. Hapus kolom customerID yang tidak relevan untuk prediksi
if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)

# 3. Ubah variabel target Churn menjadi 1 (Yes) dan 0 (No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 4. Encoding fitur kategorikal (One-Hot Encoding)
X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=True)
y = df['Churn']

# 5. Split data latih (80%) dan uji (20%) dengan Stratify
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Data Latih: {X_tr.shape}, Data Uji: {X_te.shape}")

Data Latih: (5634, 30), Data Uji: (1409, 30)


In [16]:
from sklearn.ensemble import RandomForestClassifier

# Inisialisasi dan latih model Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)
print("Pelatihan model Random Forest selesai.")

Pelatihan model Random Forest selesai.


In [17]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi label dan probabilitas
pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

# Tampilkan Laporan Evaluasi
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_te, pred, target_names=['No Churn (0)', 'Churn (1)']))

roc_auc = roc_auc_score(y_te, proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

No Churn (0)       0.83      0.89      0.86      1035
   Churn (1)       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score: 0.8246


In [18]:
# Menambahkan probabilitas churn ke dalam DataFrame hasil uji
df_hasil = X_te.copy()
df_hasil['Actual_Churn'] = y_te
df_hasil['Predicted_Prob_Churn'] = proba

# Tampilkan 5 sampel teratas dengan probabilitas tertinggi
print("Top 5 Pelanggan Berisiko Tinggi Churn:")
print(df_hasil[['Actual_Churn', 'Predicted_Prob_Churn']].sort_values(by='Predicted_Prob_Churn', ascending=False).head())

Top 5 Pelanggan Berisiko Tinggi Churn:
      Actual_Churn  Predicted_Prob_Churn
1731             1              1.000000
2194             1              0.993333
809              1              0.990000
6623             1              0.990000
1739             1              0.986667


**Kesimpulan** <br>
Modul ini membahas teknik Ensemble Learning menggunakan Random Forest untuk meningkatkan akurasi klasifikasi. Pembahasan juga berfokus pada penanganan Imbalanced Dataset menggunakan SMOTE, penyesuaian class weight, serta penentuan metrik evaluasi yang tepat seperti Recall dan PR-AUC agar model lebih andal memprediksi kelas minoritas. <br><br>

**Temuan Utama** <br>
* Random Forest menggabungkan banyak decision tree untuk menekan variance.
* Akurasi bisa menyesatkan (accuracy paradox) saat menghadapi data yang sangat tidak seimbang. <br><br>

**Pertanyaan yang Muncul** <br>
Bagaimana menentukan strategi terbaik antara penyesuaian threshold, penggunaan class weight, atau SMOTE agar performa model tetap efisien tanpa memicu overfitting?